# 🛰️ Land Classification — Full Training Pipeline
**ResNet50 | 5-Class | EuroSAT + UCM + RESISC45 → Google Maps**

## Training Strategy
| Phase | Goal | Backbone | Epochs | LR |
|---|---|---|---|---|
| **A** | Train classifier head fast | Frozen | 10 | 1e-3 |
| **B** | Fine-tune on satellite data | Top 50 layers unfrozen | 20 | 1e-4 |
| **C** | Domain adapt to Google Maps | Top 80 layers unfrozen | 15 | 5e-5 |

## Class Order (ALPHABETICAL — TF assigns these indices)
```
0 = Agriculture
1 = Bareland
2 = Urban
3 = Vegetation
4 = Water
```

## Before Running
1. **Runtime → Change runtime type → T4 GPU**
2. Upload your datasets to Google Drive under `MyDrive/LandClassification/datasets/`
3. Update the paths in **Cell 1** to match your Drive structure


In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 1 ▸ Mount Google Drive & Configure Paths
# ══════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, random, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ┌─────────────────────────────────────────────────────────┐
# │  !!! UPDATE THESE PATHS TO MATCH YOUR GOOGLE DRIVE !!!  │
# └─────────────────────────────────────────────────────────┘
DRIVE_BASE   = '/content/drive/MyDrive/LandClassification'
EUROSAT_SRC  = f'{DRIVE_BASE}/datasets/EuroSAT'
UCM_SRC      = f'{DRIVE_BASE}/datasets/UCMerced_LandUse/Images'
RESISC_SRC   = f'{DRIVE_BASE}/datasets/NWPU-RESISC45'
TEST_SET_SRC = f'{DRIVE_BASE}/datasets/test_set'   # 500 Google Maps images
MODEL_SAVE_PATH = f'{DRIVE_BASE}/models/resnet50_land_v2.keras'

# Training config
IMG_SIZE      = (224, 224)
BATCH_SIZE    = 32           # safe for T4
MAX_PER_CLASS = 9000         # cap to fix Urban class dominance
SEED          = 42
CLASS_NAMES   = ['Agriculture', 'Bareland', 'Urban', 'Vegetation', 'Water']

# Working dirs on Colab instance (faster than Drive during training)
MERGED_DIR = '/content/merged_dataset'
SPLIT_DIR  = '/content/split_dataset'

print('✅ Configuration done.')
print(f'   Classes: {CLASS_NAMES}')
print(f'   Image size: {IMG_SIZE} | Batch: {BATCH_SIZE} | Cap: {MAX_PER_CLASS}/class')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 2 ▸ Install & Verify TensorFlow
# ══════════════════════════════════════════════════════════
import tensorflow as tf
print(f'✅ TensorFlow: {tf.__version__}')
print(f'   GPU devices: {tf.config.list_physical_devices("GPU")}')
assert len(tf.config.list_physical_devices('GPU')) > 0, '❌ No GPU found! Change runtime type to T4.'

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 3 ▸ Class Mapping Definitions
# ══════════════════════════════════════════════════════════
EUROSAT_MAP = {
    'AnnualCrop': 'Agriculture', 'Forest': 'Vegetation',
    'HerbaceousVegetation': 'Vegetation', 'Highway': 'Urban',
    'Industrial': 'Urban', 'Pasture': 'Agriculture',
    'PermanentCrop': 'Agriculture', 'Residential': 'Urban',
    'River': 'Water', 'SeaLake': 'Water',
}

UCM_MAP = {
    'agricultural': 'Agriculture', 'airplane': 'Urban',
    'baseballdiamond': 'Urban', 'beach': 'Bareland',
    'buildings': 'Urban', 'chaparral': 'Vegetation',
    'denseresidential': 'Urban', 'forest': 'Vegetation',
    'freeway': 'Urban', 'golfcourse': 'Vegetation',
    'harbor': 'Water', 'intersection': 'Urban',
    'mediumresidential': 'Urban', 'mobilehomepark': 'Urban',
    'overpass': 'Urban', 'parkinglot': 'Urban',
    'river': 'Water', 'runway': 'Bareland',
    'sparseresidential': 'Urban', 'storagetanks': 'Urban',
    'tenniscourt': 'Urban',
}

RESISC_MAP = {
    'airplane': 'Urban', 'airport': 'Urban', 'baseball_diamond': 'Urban',
    'basketball_court': 'Urban', 'beach': 'Bareland', 'bridge': 'Urban',
    'chaparral': 'Vegetation', 'church': 'Urban', 'circular_farmland': 'Agriculture',
    'cloud': None,  # DISCARD
    'commercial_area': 'Urban', 'dense_residential': 'Urban', 'desert': 'Bareland',
    'forest': 'Vegetation', 'freeway': 'Urban', 'golf_course': 'Vegetation',
    'ground_track_field': 'Urban', 'harbor': 'Water', 'industrial_area': 'Urban',
    'intersection': 'Urban', 'island': 'Water', 'lake': 'Water',
    'meadow': 'Vegetation', 'medium_residential': 'Urban', 'mobile_home_park': 'Urban',
    'mountain': 'Bareland', 'overpass': 'Urban', 'palace': 'Urban',
    'parking_lot': 'Urban', 'railway': 'Urban', 'railway_station': 'Urban',
    'rectangular_farmland': 'Agriculture', 'river': 'Water', 'roundabout': 'Urban',
    'runway': 'Bareland', 'sea_ice': 'Bareland', 'ship': 'Water',
    'snowberg': 'Bareland', 'sparse_residential': 'Urban', 'stadium': 'Urban',
    'storage_tank': 'Urban', 'tennis_court': 'Urban', 'terrace': 'Agriculture',
    'thermal_power_station': 'Urban', 'wetland': 'Water',
}

print('✅ Class mappings defined.')
print(f'   EuroSAT: {len(EUROSAT_MAP)} → 5 classes')
print(f'   UCM:     {len(UCM_MAP)} → 5 classes')
print(f'   RESISC:  {len([k for k,v in RESISC_MAP.items() if v])} → 5 classes (cloud discarded)')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 4 ▸ Merge All Datasets → /content/merged_dataset/
# ══════════════════════════════════════════════════════════
def copy_files_to_class(src_folder, class_map, dest_base, dataset_name,
                         extensions=('.jpg', '.jpeg', '.png', '.tif', '.TIF')):
    counts = {c: 0 for c in CLASS_NAMES}
    src_folder = Path(src_folder)
    for orig_class, super_class in class_map.items():
        if super_class is None:
            continue
        src_dir = src_folder / orig_class
        if not src_dir.exists():
            print(f'  ⚠️  [{dataset_name}] Not found: {src_dir}')
            continue
        dest_dir = Path(dest_base) / super_class
        dest_dir.mkdir(parents=True, exist_ok=True)
        files = [f for f in src_dir.iterdir() if f.suffix.lower() in extensions]
        for f in files:
            new_name = f'{dataset_name}_{orig_class}_{f.name}'
            dest = dest_dir / new_name
            if not dest.exists():
                shutil.copy2(f, dest)
            counts[super_class] += 1
    return counts

if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)
os.makedirs(MERGED_DIR)

print('🔄 Merging EuroSAT...')
c1 = copy_files_to_class(EUROSAT_SRC, EUROSAT_MAP, MERGED_DIR, 'eurosat')
print('🔄 Merging UC Merced...')
c2 = copy_files_to_class(UCM_SRC, UCM_MAP, MERGED_DIR, 'ucm')
print('🔄 Merging RESISC45...')
c3 = copy_files_to_class(RESISC_SRC, RESISC_MAP, MERGED_DIR, 'resisc')

print(f'\n{"Class":<15} {"EuroSAT":>10} {"UCM":>8} {"RESISC45":>10} {"Total":>8}')
print('-' * 55)
totals = {}
for cls in CLASS_NAMES:
    e, u, r = c1.get(cls,0), c2.get(cls,0), c3.get(cls,0)
    totals[cls] = e+u+r
    print(f'{cls:<15} {e:>10} {u:>8} {r:>10} {totals[cls]:>8}')
print(f'\n  Grand total: {sum(totals.values())} images')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 5 ▸ Balance Classes (Cap + Compute Class Weights)
# ══════════════════════════════════════════════════════════
random.seed(SEED)
print(f'⚖️  Capping each class at {MAX_PER_CLASS} images...\n')

balanced_counts = {}
for cls in CLASS_NAMES:
    cls_dir = Path(MERGED_DIR) / cls
    all_files = list(cls_dir.iterdir())
    total = len(all_files)
    if total > MAX_PER_CLASS:
        to_remove = random.sample(all_files, total - MAX_PER_CLASS)
        for f in to_remove:
            f.unlink()
        balanced_counts[cls] = MAX_PER_CLASS
    else:
        balanced_counts[cls] = total
    print(f'  {cls:<15}: {total:>6} → {balanced_counts[cls]:>6}')

print(f'\n  Balanced total: {sum(balanced_counts.values())} images')

# Class weights — inverse frequency (Bareland gets highest boost)
max_count = max(balanced_counts.values())
CLASS_WEIGHTS = {i: round(max_count / balanced_counts[cls], 3)
                 for i, cls in enumerate(CLASS_NAMES)}
print(f'\n⚖️  Class weights:')
for i, cls in enumerate(CLASS_NAMES):
    print(f'  [{i}] {cls:<15}: {CLASS_WEIGHTS[i]}')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 6 ▸ Train / Validation Split (80/20)
# ══════════════════════════════════════════════════════════
random.seed(SEED)
if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

TRAIN_DIR = f'{SPLIT_DIR}/train'
VAL_DIR   = f'{SPLIT_DIR}/val'
VAL_SPLIT = 0.20

print(f'{"Class":<15} {"Train":>8} {"Val":>8}')
print('-' * 35)
for cls in CLASS_NAMES:
    files = list((Path(MERGED_DIR) / cls).iterdir())
    random.shuffle(files)
    n_val   = int(len(files) * VAL_SPLIT)
    for split, f_list in [('train', files[n_val:]), ('val', files[:n_val])]:
        dest = Path(SPLIT_DIR) / split / cls
        dest.mkdir(parents=True, exist_ok=True)
        for f in f_list:
            shutil.copy2(f, dest / f.name)
    print(f'  {cls:<15}: {len(files)-n_val:>8} {n_val:>8}')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 7 ▸ Data Pipeline with Augmentation
# Key: Strong color jitter to bridge SATELLITE → MAPS domain gap
# ══════════════════════════════════════════════════════════
def build_augmentation(strong=False):
    layers = [
        tf.keras.layers.RandomFlip('horizontal_and_vertical'),
        tf.keras.layers.RandomRotation(0.25),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomTranslation(0.1, 0.1),
    ]
    if strong:
        # Simulate Google Maps cartographic color palette
        layers += [
            tf.keras.layers.RandomBrightness(0.35),
            tf.keras.layers.RandomContrast(0.35),
        ]
    else:
        layers += [
            tf.keras.layers.RandomBrightness(0.20),
            tf.keras.layers.RandomContrast(0.20),
        ]
    return tf.keras.Sequential(layers)

def preprocess_resnet(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.resnet50.preprocess_input(image)
    return image, label

def load_dataset(directory, augment=False, strong_augment=False, shuffle=True):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        label_mode='categorical', class_names=CLASS_NAMES,
        shuffle=shuffle, seed=SEED,
    )
    ds = ds.cache()
    if augment:
        aug = build_augmentation(strong=strong_augment)
        ds = ds.map(lambda x, y: (aug(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(preprocess_resnet, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = load_dataset(TRAIN_DIR, augment=True,  strong_augment=False)
val_ds   = load_dataset(VAL_DIR,   augment=False, shuffle=False)
print(f'✅ Train batches: {len(train_ds)} | Val batches: {len(val_ds)}')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 8 ▸ Build ResNet50 Model
# ══════════════════════════════════════════════════════════
def build_model(num_classes=5, input_shape=(224, 224, 3)):
    base = tf.keras.applications.ResNet50(
        include_top=False, weights='imagenet',
        input_shape=input_shape, pooling=None,
    )
    base.trainable = False  # frozen for Phase A

    inputs = tf.keras.Input(shape=input_shape, name='input_image')
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = tf.keras.layers.BatchNormalization(name='bn_head')(x)
    x = tf.keras.layers.Dropout(0.4, name='drop1')(x)
    x = tf.keras.layers.Dense(
            256, activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            name='dense_head')(x)
    x = tf.keras.layers.Dropout(0.3, name='drop2')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = tf.keras.Model(inputs, outputs, name='resnet50_land_v2')
    return model, base

model, base_model = build_model()
model.summary()
total = model.count_params()
trainable = sum(np.prod(w.shape) for w in model.trainable_weights)
print(f'\n✅ {total:,} total params | {trainable:,} trainable (head only in Phase A)')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 9 ▸ PHASE A — Train Classifier Head (Backbone Frozen)
# Goal: Fast convergence of head on ~57k images
# Expected: val_accuracy ~75-82% after 10 epochs
# ══════════════════════════════════════════════════════════
PHASE_A_EPOCHS = 10

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_A = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2,
        min_lr=1e-6, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/best_phase_A.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
]

print('🚀 Phase A: Training classifier head (backbone frozen)...')
history_A = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE_A_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_A,
)
print('✅ Phase A complete!')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 10 ▸ PHASE B — Fine-tune (Top 50 Backbone Layers)
# Goal: Satellite domain adaptation
# Expected: val_accuracy ~85-91%
# ══════════════════════════════════════════════════════════
PHASE_B_EPOCHS  = 20
UNFREEZE_LAYERS = 50

base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

trainable = sum(np.prod(w.shape) for w in model.trainable_weights)
print(f'📊 Phase B: {trainable:,} trainable params ({UNFREEZE_LAYERS} layers unfrozen)')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_B = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-7, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/best_phase_B.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
]

model.load_weights('/content/best_phase_A.keras')
print('🚀 Phase B: Fine-tuning backbone...')
history_B = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE_B_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_B,
)
print('✅ Phase B complete!')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 11 ▸ PHASE C — Domain Adaptation on Google Maps
# Uses your 500 labeled Google Maps screenshots.
# HEAVY color augmentation to bridge satellite → Maps gap.
# This phase eliminates the need for the blue-pixel heuristic.
# Expected: Maps accuracy ~80-90%
# ══════════════════════════════════════════════════════════
PHASE_C_EPOCHS   = 15
UNFREEZE_PHASE_C = 80

# Split 500 Maps images → 80% fine-tune / 20% holdout
MAPS_TRAIN_DIR = '/content/maps_finetune/train'
MAPS_HOLD_DIR  = '/content/maps_finetune/holdout'

if os.path.exists('/content/maps_finetune'):
    shutil.rmtree('/content/maps_finetune')

random.seed(SEED)
print('📂 Splitting Google Maps test_set (80 fine-tune / 20 holdout):')
for cls in CLASS_NAMES:
    src = Path(TEST_SET_SRC) / cls
    if not src.exists():
        print(f'  ⚠️  Missing: {src}')
        continue
    files = list(src.iterdir())
    random.shuffle(files)
    n_hold = max(1, int(len(files) * 0.20))
    for split, f_list in [('train', files[n_hold:]), ('holdout', files[:n_hold])]:
        dest = Path(f'/content/maps_finetune/{split}') / cls
        dest.mkdir(parents=True, exist_ok=True)
        for f in f_list:
            shutil.copy2(f, dest / f.name)
    print(f'  {cls:<15}: {len(files)-n_hold} train | {n_hold} holdout')

maps_train_ds = load_dataset(MAPS_TRAIN_DIR, augment=True,  strong_augment=True)
maps_hold_ds  = load_dataset(MAPS_HOLD_DIR,  augment=False, shuffle=False)

# Unfreeze more layers for deeper domain adaptation
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_PHASE_C]:
    layer.trainable = False

# Focal Loss: penalises easily classified examples, forces model
# to focus on hard cases (Water on Maps, Bareland vs Vegetation)
def categorical_focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss(y_true, y_pred):
        eps = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        ce   = -y_true * tf.math.log(y_pred)
        wt   = alpha * y_true * tf.pow(1 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(wt * ce, axis=1))
    return focal_loss

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=categorical_focal_loss(gamma=2.0),
    metrics=['accuracy'],
)

callbacks_C = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-8, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/best_phase_C.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
]

model.load_weights('/content/best_phase_B.keras')
print('\n🚀 Phase C: Domain adaptation on Google Maps screenshots...')
history_C = model.fit(
    maps_train_ds,
    validation_data=maps_hold_ds,
    epochs=PHASE_C_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_C,
)
print('✅ Phase C complete!')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 12 ▸ Evaluate on Google Maps Holdout Set
# TRUE real-world performance metric
# ══════════════════════════════════════════════════════════
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_maps(model, holdout_dir):
    ds = tf.keras.utils.image_dataset_from_directory(
        holdout_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        label_mode='categorical', class_names=CLASS_NAMES,
        shuffle=False, seed=SEED,
    )
    ds = ds.map(preprocess_resnet, num_parallel_calls=tf.data.AUTOTUNE)
    y_true, y_pred = [], []
    for images, labels in ds:
        preds = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    print('\n' + '='*60)
    print('📊 GOOGLE MAPS HOLDOUT — CLASSIFICATION REPORT')
    print('='*60)
    print(classification_report(y_true, y_pred,
                                target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual',    fontsize=12)
    ax.set_title('Google Maps Holdout — Confusion Matrix',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/confusion_matrix_maps.png', dpi=150)
    plt.show()
    return y_true, y_pred

model.load_weights('/content/best_phase_C.keras')
y_true, y_pred = evaluate_maps(model, MAPS_HOLD_DIR)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 13 ▸ Training History Plots (All 3 Phases)
# ══════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#3498db', '#e74c3c', '#2ecc71']
labels_hist = ['Phase A (Frozen)', 'Phase B (Fine-tune)', 'Phase C (Maps Adapt)']

offset = 0
for hist, label, color in zip([history_A, history_B, history_C], labels_hist, colors):
    n = len(hist.history['accuracy'])
    epochs_range = range(offset + 1, offset + n + 1)
    ax1.plot(epochs_range, hist.history['accuracy'],
             color=color, label=f'{label} Train', linewidth=2)
    ax1.plot(epochs_range, hist.history['val_accuracy'],
             color=color, label=f'{label} Val', linewidth=2, linestyle='--')
    ax2.plot(epochs_range, hist.history['loss'],
             color=color, label=f'{label} Train', linewidth=2)
    ax2.plot(epochs_range, hist.history['val_loss'],
             color=color, label=f'{label} Val', linewidth=2, linestyle='--')
    offset += n

for ax, metric in [(ax1, 'Accuracy'), (ax2, 'Loss')]:
    ax.set_xlabel('Epoch'); ax.set_ylabel(metric)
    ax.set_title(f'{metric} — All Phases')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Land Classification Training — 3-Phase Strategy',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → /content/training_history.png')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 14 ▸ Visual Sample Predictions on Maps Images
# Green title = correct | Red title = wrong prediction
# ══════════════════════════════════════════════════════════
from PIL import Image as PILImage

n_per_class = 4
fig, axes = plt.subplots(len(CLASS_NAMES), n_per_class,
                          figsize=(n_per_class * 3, len(CLASS_NAMES) * 3))
fig.suptitle('Google Maps Predictions (Green=Correct, Red=Wrong)',
             fontsize=13, fontweight='bold')

for row, cls in enumerate(CLASS_NAMES):
    cls_dir = Path(MAPS_HOLD_DIR) / cls
    files = list(cls_dir.iterdir())[:n_per_class]
    for col, f in enumerate(files):
        img = PILImage.open(f).convert('RGB').resize(IMG_SIZE)
        arr = np.array(img, dtype=np.float32)
        inp = tf.keras.applications.resnet50.preprocess_input(arr)
        pred = model.predict(np.expand_dims(inp, 0), verbose=0)[0]
        pred_cls  = CLASS_NAMES[np.argmax(pred)]
        pred_conf = np.max(pred) * 100
        correct   = pred_cls == cls

        ax = axes[row, col]
        ax.imshow(img); ax.axis('off')
        ax.set_title(f'{pred_cls}\n{pred_conf:.1f}%',
                     color='#2ecc71' if correct else '#e74c3c',
                     fontsize=9, fontweight='bold')
        if col == 0:
            ax.set_ylabel(cls, fontsize=10)

plt.tight_layout()
plt.savefig('/content/visual_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → /content/visual_predictions.png')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 15 ▸ Save Final Model + Metadata to Google Drive
# ══════════════════════════════════════════════════════════
model.load_weights('/content/best_phase_C.keras')
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
model.save(MODEL_SAVE_PATH)
print(f'✅ Model saved → {MODEL_SAVE_PATH}')

meta = {
    'class_names':   CLASS_NAMES,
    'class_weights': CLASS_WEIGHTS,
    'img_size':      list(IMG_SIZE),
    'model_arch':    'ResNet50',
    'class_index_order': {
        '0': 'Agriculture', '1': 'Bareland',
        '2': 'Urban', '3': 'Vegetation', '4': 'Water'
    },
    'notes': (
        'Trained in 3 phases: frozen head → satellite fine-tune → Google Maps domain adaptation. '
        'No blue-pixel heuristic needed. Focal loss in Phase C handles Water/Bareland on Maps.'
    )
}
meta_path = MODEL_SAVE_PATH.replace('.keras', '_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'✅ Metadata saved → {meta_path}')

drive_plots = os.path.dirname(MODEL_SAVE_PATH)
for fname in ['confusion_matrix_maps.png', 'training_history.png', 'visual_predictions.png']:
    shutil.copy(f'/content/{fname}', f'{drive_plots}/{fname}')
    print(f'✅ Copied → {drive_plots}/{fname}')

print('\n🎉 ALL DONE!')
print(f'   📦 Model:    {MODEL_SAVE_PATH}')
print(f'   📋 Metadata: {meta_path}')
print(f'   📊 Plots:    {drive_plots}/')

# ✅ After Training — Update Your Backend

## 1. Copy `resnet50_land_v2.keras` to `backend/models/`

## 2. In `backend/app/utils.py` — change model filename:
```python
model_path = os.path.join(
    os.path.dirname(__file__), '..', 'models', 'resnet50_land_v2.keras'
)
```

## 3. Remove the blue-pixel heuristic (no longer needed!)
Delete the `_is_blue_dominant()` function and the two heuristic `if` blocks inside `classify_image()`.
The Phase C domain-adapted model handles Water on Google Maps natively.

## 4. CLASS_LABELS stays the same (alphabetical order preserved):
```python
CLASS_LABELS = [
    'Agriculture',  # index 0
    'Bareland',     # index 1
    'Urban',        # index 2
    'Vegetation',   # index 3
    'Water',        # index 4
]
```
